# to run

Run the following commands in termilan before running this notebook:
- open -a Docker
- docker run -d --name sonarqube -p 9000:9000 sonarqube:community

Run the following to reset
- docker rm -f sonarqube && docker run -d --name sonarqube -p 9000:9000 sonarqube:community

# run on single file
```
sonar-scanner \
  -Dsonar.projectKey=test-java-project \
  -Dsonar.sources=. \
  -Dsonar.host.url=http://localhost:9000 \
  -Dsonar.token=sqp_c6911ea88e86d2a621db0a09635e27adcc7e1696
```

# SonarQube Community Edition — Test Notebook

Runs a SonarQube static analysis scan on `test.java` and displays results.

## Prerequisites
1. **SonarQube server** running on `localhost:9000`  
   Quickest way: `docker run -d --name sonarqube -p 9000:9000 sonarqube:community`
2. **sonar-scanner** CLI on your PATH  
   Mac: `brew install sonar-scanner`
3. Default credentials: `admin / admin` (update in Config cell if changed)

## 1. Install Python dependencies

In [1]:
# !pip install requests pandas tabulate

## 2. Configuration

In [13]:
import os

SONAR_HOST     = "http://localhost:9000"
SONAR_TOKEN    = "sqp_30f8e1baa86b01cdcc430cc2a9879d6032f6f33d"
SONAR_LOGIN    = "admin"
SONAR_PASSWORD = ""

SONAR_PASSWORD = "Admin1111111!"      # update if you changed the default
#SONAR_PASSWORD = "admin"      # update if you changed the default 
PROJECT_KEY    = "test-java-project"
PROJECT_NAME   = "Test Java Project"

PROJECT_DIR = os.path.abspath(".")
print("Project directory:", PROJECT_DIR)

Project directory: /Users/lukas./Desktop/On-the-Naturalness-of-Agent-Generated-Documentation


## 3. Check SonarQube server is reachable

In [3]:
import requests

auth = (SONAR_TOKEN, "") if SONAR_TOKEN else (SONAR_LOGIN, SONAR_PASSWORD)

try:
    r = requests.get(f"{SONAR_HOST}/api/system/status", auth=auth, timeout=10)
    data = r.json()
    print(f"SonarQube status: {data.get('status')}  |  version: {data.get('version')}")
    if data.get("status") != "UP":
        print("WARNING: server not fully UP yet — wait a moment and re-run this cell")
except Exception as e:
    print(f"Cannot reach SonarQube at {SONAR_HOST}: {e}")
    print("Start it: docker run -d --name sonarqube -p 9000:9000 sonarqube:community")

SonarQube status: UP  |  version: 26.5.0.122743


## 4. Create the SonarQube project

In [4]:
r = requests.post(
    f"{SONAR_HOST}/api/projects/create",
    auth=auth,
    data={"project": PROJECT_KEY, "name": PROJECT_NAME}
)
if r.status_code == 200:
    print(f"Project '{PROJECT_KEY}' created.")
elif r.status_code == 400:
    msg = r.json().get("errors", [{}])[0].get("msg", "already exists")
    print(f"Project '{PROJECT_KEY}': {msg}")
else:
    print(f"Unexpected response {r.status_code}: {r.text}")

Project 'test-java-project': Could not create Project with key: "test-java-project". A similar key already exists: "test-java-project"


## 5. Generate an analysis token (skipped if SONAR_TOKEN already set)

In [5]:
if SONAR_TOKEN:
    print("Using existing SONAR_TOKEN — skipping token generation")
else:
    r = requests.post(
        f"{SONAR_HOST}/api/user_tokens/generate",
        auth=auth,
        data={"name": "notebook-token"}
    )
    if r.status_code == 200:
        SONAR_TOKEN = r.json()["token"]
        auth = (SONAR_TOKEN, "")
        print(f"Generated token (save this — shown once): {SONAR_TOKEN}")
    else:
        msg = r.json().get("errors", [{}])[0].get("msg", r.text)
        print(f"Token generation skipped ({msg}) — will use login/password")
        

Token generation skipped (A user token for login 'admin' and name 'notebook-token' already exists) — will use login/password


## 6. Write sonar-project.properties

In [14]:
props_path = os.path.join(PROJECT_DIR, "sonar-project.properties")

if SONAR_TOKEN:
    auth_line = f"sonar.token={SONAR_TOKEN}"
else:
    auth_line = f"sonar.login={SONAR_LOGIN}\nsonar.password={SONAR_PASSWORD}"

props = f"""sonar.projectKey={PROJECT_KEY}
sonar.projectName={PROJECT_NAME}
sonar.projectVersion=1.0
sonar.sources=test_proj
sonar.host.url={SONAR_HOST}
{auth_line}
"""

with open(props_path, "w") as f:
    f.write(props)

print(f"Written: {props_path}\n")
print(props)

Written: /Users/lukas./Desktop/On-the-Naturalness-of-Agent-Generated-Documentation/sonar-project.properties

sonar.projectKey=test-java-project
sonar.projectName=Test Java Project
sonar.projectVersion=1.0
sonar.sources=test_proj
sonar.host.url=http://localhost:9000
sonar.token=sqp_30f8e1baa86b01cdcc430cc2a9879d6032f6f33d



## 7. Run sonar-scanner

In [15]:
import subprocess, os, urllib.request, zipfile

SCANNER_VERSION = "5.0.1.3006"
SCANNER_EXTRACT_DIR = os.path.join(PROJECT_DIR, ".sonar-scanner-5")
SCANNER_BIN = os.path.join(SCANNER_EXTRACT_DIR, f"sonar-scanner-{SCANNER_VERSION}-macosx", "bin", "sonar-scanner")

if not os.path.exists(SCANNER_BIN):
    zip_url = (
        f"https://binaries.sonarsource.com/Distribution/sonar-scanner-cli/"
        f"sonar-scanner-cli-{SCANNER_VERSION}-macosx.zip"
    )
    zip_path = os.path.join(PROJECT_DIR, "sonar-scanner-dl.zip")
    print(f"Downloading sonar-scanner {SCANNER_VERSION} ...")
    urllib.request.urlretrieve(zip_url, zip_path)
    os.makedirs(SCANNER_EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(SCANNER_EXTRACT_DIR)
    os.remove(zip_path)
    for f in os.listdir(os.path.dirname(SCANNER_BIN)):
        os.chmod(os.path.join(os.path.dirname(SCANNER_BIN), f), 0o755)
    # Replace bundled x86_64 JRE with symlink to ARM64 Homebrew Java
    jre_path = os.path.join(SCANNER_EXTRACT_DIR, f"sonar-scanner-{SCANNER_VERSION}-macosx", "jre")
    arm_java = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
    if os.path.isdir(jre_path) and not os.path.islink(jre_path):
        os.rename(jre_path, jre_path + ".x86_64")
        os.symlink(arm_java, jre_path)
    print("Download complete.\n")
else:
    print(f"sonar-scanner {SCANNER_VERSION} already present.\n")

print("Running sonar-scanner on test_proj/ ...\n")
result = subprocess.run(
    [SCANNER_BIN, f"-Dproject.settings={props_path}"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True,
)

stdout = result.stdout
print(stdout[-4000:] if len(stdout) > 4000 else stdout)

if result.returncode != 0:
    print("\n--- STDERR ---")
    print(result.stderr[-2000:])
    print(f"\nsonar-scanner exited with code {result.returncode}")
else:
    print("\nScan completed successfully!")

sonar-scanner 5.0.1.3006 already present.

Running sonar-scanner on test_proj/ ...


INFO: No "Generated" source files to scan.
INFO: Sensor JavaSensor [java] (done) | time=464ms
INFO: Sensor JaCoCo XML Report Importer [jacoco]
INFO: 'sonar.coverage.jacoco.xmlReportPaths' is not defined. Using default locations: target/site/jacoco/jacoco.xml,target/site/jacoco-it/jacoco.xml,build/reports/jacoco/test/jacocoTestReport.xml
INFO: No report imported, no coverage information will be imported by JaCoCo XML Report Importer
INFO: Sensor JaCoCo XML Report Importer [jacoco] (done) | time=0ms
INFO: Sensor IaC hadolint report Sensor [iac]
INFO: Sensor IaC hadolint report Sensor [iac] (done) | time=0ms
INFO: Sensor Java Config Sensor [iac]
INFO: There are no files to be analyzed for the Java language
INFO: Sensor Java Config Sensor [iac] (done) | time=2ms
INFO: Sensor IaC Docker Sensor [iac]
INFO: There are no files to be analyzed for the Docker language
INFO: Sensor IaC Docker Sensor [iac] (done) |

## 8. Fetch issues from SonarQube API

In [16]:
import time, pandas as pd

time.sleep(3)  # give server time to ingest the report

def fetch_issues(project_key, page_size=500):
    issues, page = [], 1
    while True:
        r = requests.get(
            f"{SONAR_HOST}/api/issues/search",
            auth=auth,
            params={"componentKeys": project_key, "ps": page_size, "p": page, "resolved": "false"}
        )
        data = r.json()
        issues.extend(data.get("issues", []))
        if len(issues) >= data.get("total", 0):
            break
        page += 1
    return issues

raw_issues = fetch_issues(PROJECT_KEY)
print(f"Total open issues: {len(raw_issues)}")

Total open issues: 17


## 9. Display issues as a table

In [17]:
if not raw_issues:
    print("No issues returned — verify the scan completed and the project key matches.")
else:
    rows = []
    for issue in raw_issues:
        loc = issue.get("textRange", {})
        rows.append({
            "severity":  issue.get("severity"),
            "type":      issue.get("type"),
            "line":      loc.get("startLine", ""),
            "rule":      issue.get("rule"),
            "message":   issue.get("message"),
            "file":      issue.get("component", "").split(":")[-1],
        })

    df = pd.DataFrame(rows)
    df["line"] = pd.to_numeric(df["line"], errors="coerce").astype("Int64")

    sev_order = ["BLOCKER", "CRITICAL", "MAJOR", "MINOR", "INFO"]
    df["severity"] = pd.Categorical(df["severity"], categories=sev_order, ordered=True)
    df = df.sort_values(["severity", "line"]).reset_index(drop=True)

    pd.set_option("display.max_colwidth", 80)
    pd.set_option("display.max_rows", 200)
    display(df)

,severity,type,line,rule,message,file
0,BLOCKER,BUG,55,java:S2095,"Use try-with-resources or close this ""FileInputStream"" in a ""finally"" clause.",test_proj/test.java
1,CRITICAL,CODE_SMELL,23,java:S3776,Refactor this method to reduce its Cognitive Complexity from 22 to the 15 al...,test_proj/test.java
2,MAJOR,CODE_SMELL,7,java:S1068,"Remove this unused ""PASSWORD"" private field.",test_proj/test.java
3,MAJOR,VULNERABILITY,7,java:S2068,"'PASSWORD' detected in this expression, review this potentially hard-coded p...",test_proj/test.java
4,MAJOR,CODE_SMELL,8,java:S1068,"Remove this unused ""DB_URL"" private field.",test_proj/test.java
5,MAJOR,CODE_SMELL,12,java:S106,Replace this use of System.out by a logger.,test_proj/test.java
6,MAJOR,CODE_SMELL,14,java:S1854,"Remove this useless assignment to local variable ""result"".",test_proj/test.java
7,MAJOR,CODE_SMELL,57,java:S106,Replace this use of System.out by a logger.,test_proj/test.java
8,MAJOR,CODE_SMELL,80,java:S4144,"Update this method so that its implementation is not identical to ""sumA"" on ...",test_proj/test.java
9,MAJOR,CODE_SMELL,96,java:S1854,"Remove this useless assignment to local variable ""x"".",test_proj/test.java


## 10. Summary by severity and type

In [10]:
if raw_issues:
    print("=== By Severity ===")
    display(df.groupby("severity", observed=True).size().rename("count").reset_index())

    print("\n=== By Type ===")
    display(df.groupby("type").size().rename("count").reset_index())

    print("\n=== Top Rules ===")
    top = df["rule"].value_counts().head(10).reset_index()
    top.columns = ["rule", "count"]
    display(top)

## 11. Project quality metrics

In [11]:
metric_keys = [
    "bugs", "vulnerabilities", "code_smells", "security_hotspots",
    "coverage", "duplicated_lines_density", "ncloc",
    "sqale_index", "reliability_rating", "security_rating", "sqale_rating"
]

r = requests.get(
    f"{SONAR_HOST}/api/measures/component",
    auth=auth,
    params={"component": PROJECT_KEY, "metricKeys": ",".join(metric_keys)}
)

measures = r.json().get("component", {}).get("measures", [])
metrics_df = pd.DataFrame([
    {"metric": m["metric"], "value": m.get("value", "N/A")}
    for m in measures
])
display(metrics_df)

""


## 12. Open dashboard in browser

In [12]:
import webbrowser
url = f"{SONAR_HOST}/dashboard?id={PROJECT_KEY}"
print(f"Opening: {url}")
webbrowser.open(url)

Opening: http://localhost:9000/dashboard?id=test-java-project


True